# 00_12 — Comparación numérica MATLAB ↔ Python (PSBP-FD v2)

**Objetivo:** validar que la implementación Python (`PSBPSampler` + `PSBPPredictor` orquestados por `PSBP_FD_v2`) reproduce, en distribución posterior, los resultados del código MATLAB original de Chung & Dunson (2009) sobre los datos `BHPin_1.txt` / `BHPout_1.txt`.

Aqui solo ejecutremos los codigos de python nada mas.

## 1. Imports, rutas y carga del `.mat`

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os, sys, time
from scipy.io import loadmat
from scipy.stats import norm

# ── Raíz del proyecto ──────────────────────────────────────────────────
def get_project_root(marker: str = "README.md") -> Path:
    current = Path(os.getcwd()).resolve()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    return current

PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

# ── Rutas ──────────────────────────────────────────────────────────────
PATHS = {
    "raw":      PROJECT_ROOT / "data" / "simulaciones" / "raw",
    "mat":      PROJECT_ROOT / "data" / "simulaciones" / "raw",  # poner BHP_1.mat aquí
    "reports":  PROJECT_ROOT / "reports" / "simulaciones" / "matlab_vs_python",
}
PATHS["reports"].mkdir(parents=True, exist_ok=True)

BASEFNAME = "BHP"
TT        = 1
SEED      = 42  # arbitraria; las trazas individuales no son comparables igual

print(f"\nRutas:")
for k, v in PATHS.items():
    print(f"  {k:8s} → {v}")

PROJECT_ROOT: C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd

Rutas:
  raw      → C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\data\simulaciones\raw
  mat      → C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\data\simulaciones\raw
  reports  → C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\reports\simulaciones\matlab_vs_python


## 2. Carga directa de `BHPin_1.txt` / `BHPout_1.txt` y estandarización

In [6]:
# ── Carga de los .txt crudos ───────────────────────────────────────────
path_in  = PATHS["raw"] / f"{BASEFNAME}in_{TT}.txt"
path_out = PATHS["raw"] / f"{BASEFNAME}out_{TT}.txt"

dt_train = np.loadtxt(path_in)    # (n,  1+p) — col 0 es y
dt_test  = np.loadtxt(path_out)   # (n2, 1+p)

n,  p1 = dt_train.shape
n2, _  = dt_test.shape
p      = p1 - 1
print(f"Train: n  = {n},  p = {p}")
print(f"Test:  n2 = {n2}, p = {p}")

# ── Estandarización con MATLAB ddof=0 ──────────────────────────────────
#  IMPORTANTE: usar np.std(..., ddof=0) — el default de pandas es ddof=1
dt_mean = dt_train.mean(axis=0)
dt_std  = dt_train.std(axis=0, ddof=0)

dt_train_std = (dt_train - dt_mean) / dt_std
dt_test_std  = (dt_test  - dt_mean) / dt_std

# ── Construir DataFrame de entrenamiento ──────────────────────────────
df_train = pd.DataFrame(
    dt_train_std,
    columns=["y"] + [f"x{j+1}" for j in range(p)],
)
df_test = pd.DataFrame(
    dt_test_std,
    columns=["y"] + [f"x{j+1}" for j in range(p)],
)
print(f"\ndf_train shape: {df_train.shape}")
print(df_train.head(3))

Train: n  = 249,  p = 13
Test:  n2 = 125, p = 13

df_train shape: (249, 14)
          y        x1        x2        x3        x4        x5        x6  \
0 -0.894565 -0.557821  0.695499 -0.432297 -0.295527 -0.741107 -1.009233   
1  0.414807 -0.559309  0.620056 -1.040311 -0.295527 -0.415497  0.675903   
2  2.443744 -0.581453  0.129677 -0.860101 -0.295527 -0.693364  2.107457   

         x7        x8        x9       x10       x11       x12       x13  
0 -1.242294  1.052115 -2.121417 -0.154098 -0.384493  0.338922  0.240811  
1  0.289954 -0.558429  1.576829 -1.178978  0.290357  0.408415 -0.553579  
2  0.077853  0.145776  0.344080 -1.253969 -1.284292  0.177013 -1.164773  


## 4. Ejecutar `PSBP_FD_v2` con la misma configuración MCMC del `.m`

| Hiperparámetro | Valor MATLAB | Valor Python |
|---|---|---|
| nsim | 2000 | 2000 |
| burn | 200 | 200 |
| N (trunc.) | 20 | 20 |
| M (grilla) | 50 | 50 |
| atau, btau | 0.5, 0.5 | 0.5, 0.5 |
| ag, bg | 0.5, 0.5 | 0.5, 0.5 |
| apij, bpij | 1, 5 | 1, 5 |
| mumu, taumu | 0, 1 | 0, 1 |
| mupsij, taupsij | 0, 1 | 0, 1 |
| pwj | 0.5 | 0.5 |

In [ ]:
from model_psbp_fd.models.psbp_fd_v2.psbp_fd_v2 import PSBP_FD_v2

MCMC_CFG = {"nsim": 2000, "burn": 200, "N": 20, "M": 50}
HP = {
    "atau":  0.5, "btau":  0.5,
    "ag":    0.5, "bg":    0.5,
    "apij":  1.0, "bpij":  5.0,
    "mumu":  0.0, "taumu": 1.0,
    "mupsij":  0.0, "taupsij": 1.0,
    "pwj":   0.5,
}

print("Ajustando PSBP_FD_v2 ...")
t0 = time.perf_counter()
model = PSBP_FD_v2(
    mcmc_cfg=MCMC_CFG, hp=HP,
    seed=SEED, verbose_every=200,
).fit(df_train)
elapsed = time.perf_counter() - t0
print(f"\nFit completado en {elapsed:.1f}s ({elapsed/60:.2f} min).")

traces = model.traces
print(f"\nClaves de trazas Python: {sorted(traces.keys())}")

Ajustando PSBP_FD_v2 ...


C:\Users\JuanFran\Desktop\git_tesis\model_psbp_fd\model_psbp_fd\models\psbp_fd_v2\functions\sampler.py:79: RuntimeWarning: covariance is not symmetric positive-semidefinite.
  return rng.multivariate_normal(mu.ravel(), cov)


## 5. RMSE

In [ ]:
# ── RMSE Python en escala estandarizada ────────────────────────────────
rmse_py_in_std  = model.rmse(df_train)
rmse_py_out_std = model.rmse(df_test)


## 6. Probabilidades de inclusión por variable



In [ ]:
## Aqui impirmir probabilidades de inclusion y grafica 

## 7. Scaterplot Real vs Predicho

## 8. Distribuciones marginales posteriores de hiperparámetros 
- Analizar trazas 
- Ver ACF 
- Ver distribucion 